# 01 — Exploratory Data Analysis: Brand24/mms

This notebook:
1. Loads the `Brand24/mms` dataset from HuggingFace
2. Prints the schema and per-language sample counts
3. Plots the label distribution per language as a stacked bar chart
4. Checks for a parallel-sample ID column
5. Saves `results/scores/label_distribution.csv`

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on the path
REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))
print('Repo root:', REPO_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from src.data.loader import LABEL_MAP, load_mms

SCORES_DIR = REPO_ROOT / 'results' / 'scores'
SCORES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load dataset and inspect schema

In [ ]:
import os

# Load HF_TOKEN from .env if present (never hardcode credentials)
_env = REPO_ROOT / ".env"
if _env.exists():
    for _line in _env.read_text().splitlines():
        if "=" in _line and not _line.startswith("#"):
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())

dataset = load_mms(cache_dir=str(REPO_ROOT / "data" / "raw"))


In [ ]:
# Show all splits
print('Splits:', list(dataset.keys()))
first_split = next(iter(dataset.values()))
print('\nFeatures:')
for col, feat in first_split.features.items():
    print(f'  {col:30s}  {feat}')

In [ ]:
# Show the first 3 rows of the first split
first_split.to_pandas().head(3)

## 2. Per-language sample counts

In [ ]:
# Combine all splits into one DataFrame for EDA
dfs = []
for split_name, split_ds in dataset.items():
    df = split_ds.to_pandas()
    df['split'] = split_name
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print('Total rows:', len(df_all))
print('Columns:', list(df_all.columns))
df_all.head(3)

In [ ]:
# Detect language and text columns
LANG_CANDIDATES = ['language', 'lang', 'Language']
TEXT_CANDIDATES = ['text', 'sentence', 'content', 'review']

lang_col = next((c for c in LANG_CANDIDATES if c in df_all.columns), None)
text_col = next((c for c in TEXT_CANDIDATES if c in df_all.columns), None)
label_col = 'label' if 'label' in df_all.columns else None

print(f'Language column : {lang_col}')
print(f'Text column     : {text_col}')
print(f'Label column    : {label_col}')

In [ ]:
if lang_col:
    lang_counts = df_all[lang_col].value_counts().sort_index()
    print(f'Number of languages: {len(lang_counts)}')
    print(lang_counts.to_string())

## 3. Label distribution per language

In [ ]:
if lang_col and label_col:
    # Map integer labels to names
    df_all['label_name'] = df_all[label_col].map(LABEL_MAP)

    # Build pivot table: rows = language, cols = label
    label_dist = (
        df_all.groupby([lang_col, 'label_name'])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=['negative', 'neutral', 'positive'], fill_value=0)
    )
    label_dist_norm = label_dist.div(label_dist.sum(axis=1), axis=0)

    print('Label distribution (counts):')
    display(label_dist)
else:
    print('Could not find language or label column — check dataset schema.')

In [ ]:
if lang_col and label_col:
    fig, ax = plt.subplots(figsize=(14, 6))
    label_dist_norm.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=['#d73027', '#fee090', '#4dac26'],
        edgecolor='white',
        linewidth=0.4,
    )
    ax.set_title('Normalised Label Distribution per Language — Brand24/mms', fontsize=13)
    ax.set_xlabel('Language', fontsize=11)
    ax.set_ylabel('Fraction', fontsize=11)
    ax.legend(title='Sentiment', bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    fig.tight_layout()
    plt.show()

## 4. Check for parallel-sample ID column

In [ ]:
ID_CANDIDATES = ['id', 'sample_id', 'parallel_id', 'doc_id', 'idx']
id_col = next((c for c in ID_CANDIDATES if c in df_all.columns), None)

if id_col:
    print(f'Parallel ID column found: "{id_col}"')
    unique_ids = df_all[id_col].nunique()
    print(f'  Unique IDs       : {unique_ids}')
    print(f'  Total rows       : {len(df_all)}')
    if lang_col:
        counts_per_id = df_all.groupby(id_col)[lang_col].nunique()
        fully_parallel = (counts_per_id == df_all[lang_col].nunique()).sum()
        print(f'  Fully parallel IDs (all languages): {fully_parallel}')
else:
    print('No parallel ID column detected among candidates:', ID_CANDIDATES)
    print('All available columns:', list(df_all.columns))
    print()
    print('ACTION: Inspect the columns above and update build_parallel_samples()')
    print('        in src/data/loader.py once the correct column is identified.')

## 5. Save label_distribution.csv

In [ ]:
if lang_col and label_col:
    out_path = SCORES_DIR / 'label_distribution.csv'
    label_dist.to_csv(out_path)
    print(f'Saved → {out_path}')
else:
    print('Skipped: language or label column not found.')